# **Demo dịch ngôn ngữ ký hiệu — XE vs SCST**

Video ký hiệu → **MediaPipe Holistic** trích pose 183-d mỗi frame → **Transformer encoder-decoder** sinh câu tiếng Đức.
Cùng một chuỗi pose chạy qua **2 checkpoint** của cùng một run:

| Checkpoint | Ý nghĩa |
|---|---|
| `best_xe.pt` | kết thúc pha Cross-Entropy (baseline) |
| `last_rl.pt` | kết thúc pha RL/SCST (tối ưu thẳng BLEU bằng policy gradient) |

**Chuẩn bị trước khi chạy** — upload lên Google Drive một thư mục (vd `MyDrive/slt_demo/`) gồm:

```
slt_demo/
  run1_transformer_subset25/     <- copy từ results/phoenix/
      best_xe.pt                 (~43 MB)
      last_rl.pt                 (~43 MB)
  spm.model                      <- copy từ evidence/phoenix25/spm.model
  slt_demo.py                    <- 2 file trong demo/ của repo này
  app.py
```

> **Runtime**: GPU không bắt buộc (model chỉ 10M tham số, CPU decode <1s/câu). Nút cổ chai là MediaPipe — vốn **chỉ chạy CPU**. Chọn `T4 GPU` nếu có sẵn, không có thì `CPU` vẫn dùng tốt.

> **Lưu ý về chất lượng**: model train trên **PHOENIX-2014T** (bản tin thời tiết tiếng Đức, một người ký chính diện, nền xám, 210×260px), test BLEU-4 ≈ 6. Video quay bằng webcam ở bối cảnh khác nằm **ngoài phân phối train** → câu sinh ra sẽ lảm nhảm về thời tiết. Đó là hành vi đúng của model chứ không phải lỗi demo. Muốn xem output sát thực tế, dùng tab **Pose .npz có sẵn** với một file từ `phoenix-poses`.

## **Cell 1 — Cài dependencies**

`mediapipe==0.10.14` là bản đã dùng khi trích pose cho toàn corpus (`Sign-Language-REL_pose-extract.ipynb`) — giữ nguyên để pose lúc demo khớp với pose lúc train. Từ 0.10.31 MediaPipe **gỡ hẳn** `mediapipe.solutions.holistic` nên không dùng bản mới được.

In [ ]:
!pip uninstall -y -q mediapipe mediapipe-nightly 2>/dev/null
!pip install -q "mediapipe==0.10.14" || pip install -q "mediapipe<=0.10.21"
!pip install -q sentencepiece gradio opencv-python-headless

import mediapipe, sys
print("mediapipe", mediapipe.__version__, "| python", sys.version.split()[0])
try:
    mediapipe.solutions.holistic
    print("solutions.holistic OK")
except AttributeError:
    print("top-level solutions thiếu — slt_demo.py sẽ tự import submodule, không sao")

## **Cell 2 — Lấy code train**

Demo **không chép lại** kiến trúc model mà import thẳng `models/`, `configs/`, `data/tokenizer.py` từ repo train. Chép lại rất dễ lệch một tham số (d_model, LayerNorm cuối decoder, weight tying) — khi đó `load_state_dict` vẫn chạy nhưng output thành rác.

In [ ]:
import os
os.chdir("/content")
!rm -rf /content/Sign-Language-REL_code
!git clone -q https://github.com/linhxm/Sign-Language-REL_code.git

CODE_DIR = "/content/Sign-Language-REL_code"
os.environ["SLT_CODE_DIR"] = CODE_DIR
assert os.path.isfile(f"{CODE_DIR}/models/slt_transformer.py"), "clone hỏng?"
print("code:", CODE_DIR)

## **Cell 3 — Mount Drive & trỏ đường dẫn**

Sửa `DRIVE_DIR` cho khớp thư mục bạn đã upload.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/slt_demo"   # <<< SỬA CHỖ NÀY
RESULTS_DIR = DRIVE_DIR                          # thư mục cha chứa các run
SPM_PATH = os.path.join(DRIVE_DIR, "spm.model")

assert os.path.isdir(DRIVE_DIR), f"Không thấy {DRIVE_DIR} — kiểm tra lại đường dẫn trên Drive"
print(sorted(os.listdir(DRIVE_DIR)))

## **Cell 4 — Lấy 2 file demo (`slt_demo.py`, `app.py`)**

Thử lần lượt: có sẵn trong repo code đã clone → có trên Drive → cuối cùng mới hỏi upload tay.

In [ ]:
import shutil

DEMO_DIR = "/content/demo"
os.makedirs(DEMO_DIR, exist_ok=True)
NEEDED = ["slt_demo.py", "app.py"]

for fname in NEEDED:
    dst = os.path.join(DEMO_DIR, fname)
    if os.path.exists(dst):
        continue
    for src in (os.path.join(CODE_DIR, "demo", fname), os.path.join(DRIVE_DIR, fname)):
        if os.path.exists(src):
            shutil.copy(src, dst)
            print("copied", src)
            break

missing = [f for f in NEEDED if not os.path.exists(os.path.join(DEMO_DIR, f))]
if missing:
    print("Thiếu", missing, "-> chọn file để upload:")
    from google.colab import files
    for name, data in files.upload().items():
        open(os.path.join(DEMO_DIR, name), "wb").write(data)

import sys
sys.path.insert(0, DEMO_DIR)
print(sorted(os.listdir(DEMO_DIR)))

## **Cell 5 — Kiểm tra checkpoint tìm được**

Nạp thử model ngay ở đây để nếu có lỗi (sai đường dẫn, lệch vocab) thì thấy lỗi rõ ràng, thay vì lỗi chìm bên trong callback của Gradio.

In [ ]:
import torch
from slt_demo import discover_runs, Translator

runs = discover_runs(RESULTS_DIR)
assert runs, f"Không thấy best_xe.pt nào dưới {RESULTS_DIR}"
for name, paths in runs.items():
    print(name, "->", {k: os.path.basename(v) for k, v in paths.items()})

RUN = sorted(runs)[0]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
xe = Translator.from_checkpoint(runs[RUN]["xe"], SPM_PATH, device=DEVICE, name="XE")
print(f"\n[{RUN}] nạp OK · device={DEVICE} · vocab={xe.tokenizer.vocab_size} · meta={xe.meta}")

## **Cell 6 — Chạy app**

`share=True` tạo link `*.gradio.live` mở được ngoài Colab (sống ~72h). Lần đầu bấm **Dịch** sẽ mất thêm ~10s để MediaPipe khởi tạo.

In [ ]:
import app as demo_app

demo_app.STATE["runs"] = runs
demo_app.STATE["spm"] = SPM_PATH
demo_app.STATE["device"] = DEVICE

demo_app.build_ui().launch(share=True, debug=False)

## **Cell 7 (tuỳ chọn) — Dịch bằng script, không cần giao diện**

Hữu ích khi cần chạy hàng loạt video, hoặc khi link Gradio bị chặn.

In [ ]:
from slt_demo import PoseExtractor, Translator

VIDEO = "/content/test.mp4"   # <<< đường dẫn video của bạn

pose, stats, _ = PoseExtractor().extract(VIDEO)
print(f"pose {pose.shape} · detect tay trái/phải: "
      f"{stats.det_lhand*100:.0f}% / {stats.det_rhand*100:.0f}% · {stats.extract_s:.1f}s\n")

for label, key in [("XE  ", "xe"), ("SCST", "scst")]:
    if key not in runs[RUN]:
        continue
    tr = Translator.from_checkpoint(runs[RUN][key], SPM_PATH, device=DEVICE, name=label)
    text, dt = tr.translate(pose, decode="beam", beam_size=4)
    print(f"{label}: {text}   ({dt*1000:.0f} ms)")